In [ ]:
# Cell 0
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from datetime import datetime, timedelta

# Configurar Spark com conector BigQuery
spark = SparkSession.builder \
    .appName("PERF_AMIGAVEL_GCP") \
    .config("spark.jars", "gs://spark-lib/bigquery/spark-bigquery-latest_2.12.jar") \
    .getOrCreate()

In [ ]:
# Cell 1
DATA_1 = '20251023';
DATA_2 = '20251022';
DATA_3 = '20250930';

In [ ]:
# Cell 2
DT_ARQ = datetime.strptime(DATA_1, '%Y%m%d')
SAFRA = datetime.strptime(DATA_3, '%Y%m%d')
DTINICIOMES = DT_ARQ.replace(day=1)
ANOMESCARGA = DT_ARQ.strftime('%Y%m')

DTINICIO = datetime.strptime(DATA_3, '%Y%m%d').strftime('%Y-%m-%d %H:%M:%S')
DTFIM = datetime.strptime(DATA_1, '%Y%m%d').strftime('%Y-%m-%d %H:%M:%S')
DATA_REG = DT_ARQ.strftime('%Y-%m-%d')

print(f"DTINICIO: {DTINICIO}")
print(f"DTFIM: {DTFIM}")
print(f"DATA_REG: {DATA_REG}")

In [ ]:
# Cell 3
bq_project = 'bv-sadi-dtm'
bq_schema = 'bv_pdbd_prd'
bq_credfile = 'credfile.json'

In [ ]:
# Cell 4
df_base_gcp = spark.read \
    .format("bigquery") \
    .option("project", bq_project) \
    .option("dataset", bq_schema) \
    .option("credentialsFile", bq_credfile) \
    .option("query", f"""
        WITH QUERY_1 AS (
            SELECT
                a.dt_disponibilizacao AS DTENVIO,
                DATE_ADD(DATE(a.dt_disponibilizacao), INTERVAL 1 DAY) AS DTENVIO_REAL,
                a.dt_nova_disponib AS DT_NOVA_DISPONIB,
                DATE_ADD(DATE(a.dt_nova_disponib), INTERVAL 1 DAY) AS DT_NOVA_DISPONIB_REAL,
                a.cd_nova_assessoria AS NOVA_ASSESSORIA,
                a.cd_novo_status AS NOVO_STATUS,
                CAST(a.nr_contrato AS INT64) AS CHAVE,
                CAST(SUBSTR(a.nr_contrato, 4) AS INT64) * 1 AS NUCONTA,
                CAST(SUBSTR(a.nr_contrato, 1, 3) AS INT64) * 1 AS PRODUTO,
                CAST(a.nr_dias_atraso AS INT64) * 1 AS DIAS_AT,
                CAST(d.dmdays AS INT64) * 1 AS ATRASO_ATU,
                a.score1 AS CS,
                CAST(c.u3codvar AS INT64) AS DIG_SCORE,
                CAST(d.dmsubset AS INT64) AS DIG_RAND,
                a.dmque2 AS FAIXA_CS,
                CAST(a.cd_assessoria AS INT64) AS COD_ASSESS,
                CASE
                    WHEN CAST(a.cd_assessoria AS INT64) = 4001 THEN 'TOLEDO'
                    WHEN CAST(a.cd_assessoria AS INT64) = 4002 THEN 'REALE'
                    WHEN CAST(a.cd_assessoria AS INT64) = 4003 THEN 'ZANC'
                    WHEN CAST(a.cd_assessoria AS INT64) = 4005 THEN 'PASCHOALOTTO'
                    WHEN CAST(a.cd_assessoria AS INT64) = 4006 THEN 'BELLINATI'
                    WHEN CAST(a.cd_assessoria AS INT64) = 4007 THEN 'CRC'
                    WHEN CAST(a.cd_assessoria AS INT64) = 4008 THEN 'TOLEDO_2'
                    WHEN CAST(a.cd_assessoria AS INT64) = 4009 THEN 'BELLINATI_2'
                    WHEN CAST(a.cd_assessoria AS INT64) = 4010 THEN 'PILOTO_BOM'
                    WHEN CAST(a.cd_assessoria AS INT64) = 4062 THEN 'CONCILIG'
                END AS ASSESSORIA,
                a.dmque AS STATUS_CONTA,
                CAST(a.vl_divida AS NUMERIC) AS SALDO_DEVEDOR_VENC,
                CAST(d.dmcurbal AS NUMERIC) AS DMCURBAL,
                CAST(a.vl_nova_divida AS NUMERIC) AS VL_NOVA_DIVIDA,
                c.u3dtpriatr AS DT_PRI_ATRASO,
                c.u3dtenqcont AS DT_ENQUADRAMENTO,
                CAST(c.u3score4 AS INT64) AS U3SCORE4
            FROM
                `{bq_project}.{bq_schema}.trusted_rcvry.tb_historico_disponibilizacao` AS a
            LEFT JOIN
                `{bq_project}.{bq_schema}.trusted_rcvry.uda3` AS c
            ON
                a.nr_contrato = c.u3acct
            LEFT JOIN
                `{bq_project}.{bq_schema}.trusted_rcvry.delqmst` AS d
            ON
                a.nr_contrato = d.dmacct
            WHERE
                CAST(a.nr_grupo AS INT64) = 3
                AND a.dmque = 'CA'
                AND a.dmque2 IN ('BS', 'REG', 'RUIM', 'BOM')
                AND CAST(a.cd_assessoria AS INT64) IN (4001, 4002, 4003, 4005, 4006, 4007, 4008, 4009, 4010, 4062)
                AND CAST(c.u3cdleg AS INT64) <> 2
                AND (
                    (a.dt_disponibilizacao >= TIMESTAMP('{DTINICIO}') AND a.dt_disponibilizacao < TIMESTAMP('{DTFIM}'))
                    OR
                    (a.dt_disponibilizacao < TIMESTAMP('{DTINICIO}') AND (a.dt_nova_disponib IS NULL OR a.dt_nova_disponib > TIMESTAMP('{DTINICIO}')))
                )
                AND a.dt_disponibilizacao >= PARSE_TIMESTAMP('%Y/%m/%d %H:%M:%S', '2019/01/01 23:59:59')
        )
        SELECT * FROM QUERY_1
    """) \
    .load()

df_base_gcp.createOrReplaceTempView(f"BASE_GCP_{DATA_1}")

In [ ]:
# Cell 5
spark.sql(f"""
    CREATE OR REPLACE TABLE BASE_CYBER_{DATA_1} AS 
    SELECT DISTINCT
        DATE_TRUNC('MONTH', DATE('{DT_ARQ.strftime('%Y-%m-%d')}')) AS SAFRA,
        DATE('{DT_ARQ.strftime('%Y-%m-%d')}') AS DT_ARQUIVO,
        A.DTENVIO,
        DATE(A.DT_NOVA_DISPONIB) AS DT_NOVA_DISPONIB,
        CASE
            WHEN C.DT_ENVIO_REAL IS NOT NULL THEN C.DT_ENVIO_REAL 
            ELSE A.DT_NOVA_DISPONIB_REAL
        END AS DT_NOVA_DISPONIB_REAL,
        A.NOVA_ASSESSORIA,
        A.NOVO_STATUS,
        A.DTENVIO_REAL as DTENVIO_REAL_OLD, 
        CASE
            WHEN B.DT_ENVIO_REAL IS NOT NULL THEN B.DT_ENVIO_REAL 
            ELSE A.DTENVIO_REAL
        END AS DTENVIO_REAL,
        A.CHAVE,
        A.NUCONTA,
        A.PRODUTO,
        A.DIAS_AT,
        A.ATRASO_ATU,
        DATEDIFF(DATE('{DTINICIOMES.strftime('%Y-%m-%d')}'), DATE(A.DT_PRI_ATRASO)) as ATRASO_INI_MES,
        A.DT_PRI_ATRASO,
        A.CS,
        A.DIG_SCORE,
        A.DIG_RAND,
        A.FAIXA_CS,
        A.COD_ASSESS,
        A.ASSESSORIA,
        A.STATUS_CONTA,
        A.SALDO_DEVEDOR_VENC,
        A.DMCURBAL,
        A.VL_NOVA_DIVIDA,
        A.DT_ENQUADRAMENTO,
        A.U3SCORE4
    FROM BASE_GCP_{DATA_1} A 
    LEFT JOIN ATIVA.FIM_DE_SEMANA B
        ON A.DTENVIO_REAL = B.FIM_DE_SEMANA
    LEFT JOIN ATIVA.FIM_DE_SEMANA C
        ON A.DT_NOVA_DISPONIB_REAL = C.FIM_DE_SEMANA
""")

In [ ]:
# Cell 6
spark.sql(f"""
    CREATE OR REPLACE TABLE BASE_CYBER_{DATA_1} AS
    SELECT 
        *,
        -- Marcação de Fluxo ou Estoque
        CASE 
            WHEN DTENVIO_REAL >= DATE('{DTINICIOMES.strftime('%Y-%m-%d')}') THEN 'F'
            ELSE 'E'
        END AS TPCLASSE,
        
        -- Correção da Faixa CS
        CASE 
            WHEN FAIXA_CS = 'BS' THEN 'BAIXO_SALDO'
            WHEN FAIXA_CS = 'RUIM' THEN 'RUIM'
            WHEN FAIXA_CS = 'REG' THEN 'REGULAR'
            WHEN FAIXA_CS = 'BOM' THEN 'BOM'
        END AS FAIXA_CS_ATU,
        
        -- Piloto Digital
        CASE 
            WHEN COD_ASSESS = 4010 THEN 'S'
            ELSE 'N'
        END AS PILOTO_DIGITAL,
        
        -- Conta Digital
        CASE 
            WHEN COD_ASSESS = 4007 
                AND DTENVIO_REAL >= DATE('2021-03-02') 
                AND DIG_SCORE IN (31,32,33,34) THEN 'S'
            ELSE 'N'
        END AS CONTA_DIGITAL
        
    FROM BASE_CYBER_{DATA_1}
""")

spark.sql(f"""
    CREATE OR REPLACE TABLE BASE_CYBER_{DATA_1} AS
    SELECT 
        * EXCEPT(ASSESSORIA),
        CASE 
            WHEN ASSESSORIA = "TOLEDO_2" THEN "TOLEDO"
            WHEN ASSESSORIA = "BELLINATI_2" THEN "BELLINATI"
            ELSE ASSESSORIA
        END AS ASSESSORIA
    FROM BASE_CYBER_{DATA_1}
""")

In [ ]:
# Cell 7
spark.sql(f"""
            WITH Enquadramento_Filtrado AS (
                SELECT
                    NUCONTA,
                    PRODUTO,
                    CAST(DT_ENQUADRAMENTO AS DATE) AS DATA_ENQUAD
                FROM BASE_CYBER_{DATA_1}
                WHERE DT_ENQUADRAMENTO IS NOT NULL)

            INSERT INTO MOVTO.BASE_HIST_ENQUAD
            SELECT
                A.NUCONTA,
                A.PRODUTO,
                A.DATA_ENQUAD
            FROM Enquadramento_Filtrado AS A
            LEFT ANTI JOIN MOVTO.BASE_HIST_ENQUAD AS B
                ON A.NUCONTA = B.NUCONTA
                AND A.PRODUTO = B.PRODUTO;
            """)


spark.sql(f"""
        CREATE OR REPLACE TABLE MOVTO.BASE_HIST_ENQUAD AS
        SELECT
            NUCONTA,
            PRODUTO,
            MIN(DATA_ENQUAD) AS DATA_ENQUAD
        FROM MOVTO.BASE_HIST_ENQUAD
        GROUP BY
            NUCONTA,
            PRODUTO;
    """)

In [ ]:
# Cell 8
spark.sql(f"""
            CREATE TABLE WORK.FILTER_FOR_BASE_CYBER_2025 AS 
            SELECT t1.SAFRA, 
                    t1.DT_ARQUIVO, 
                    t1.DTENVIO, 
                    t1.DT_NOVA_DISPONIB, 
                    t1.DT_NOVA_DISPONIB_REAL, 
                    t1.NOVA_ASSESSORIA, 
                    t1.NOVO_STATUS, 
                    t1.DTENVIO_REAL_OLD, 
                    t1.DTENVIO_REAL, 
                    t1.CHAVE, 
                    t1.NUCONTA, 
                    t1.PRODUTO, 
                    t1.DIAS_AT, 
                    t1.ATRASO_ATU, 
                    t1.ATRASO_INI_MES, 
                    t1.DT_PRI_ATRASO, 
                    t1.CS, 
                    t1.DIG_SCORE, 
                    t1.DIG_RAND, 
                    t1.FAIXA_CS, 
                    t1.COD_ASSESS, 
                    t1.ASSESSORIA, 
                    t1.STATUS_CONTA, 
                    t1.SALDO_DEVEDOR_VENC, 
                    t1.DMCURBAL, 
                    t1.VL_NOVA_DIVIDA, 
                    t1.DT_ENQUADRAMENTO, 
                    t1.U3SCORE4, 
                    t1.TPCLASSE, 
                    t1.FAIXA_CS_ATU, 
                    t1.PILOTO_DIGITAL, 
                    t1.CONTA_DIGITAL
                FROM WORK.BASE_CYBER_20250401 t1
                ORDER BY t1.DTENVIO;
                """)

In [ ]:
# Cell 9
basileia_path = f"/riscosbv/cobranca/erocha/CARTAO/MOVTO/CYBER/BASILEIA_DAILY_BALANCE_1001_{DATA_1}.DAT"

df_basileia_raw = spark.read.text(basileia_path)

# Parsing dos campos conforme posições do SAS INPUT
from pyspark.sql.functions import substring, trim, col, when, lit

df_basileia = df_basileia_raw.select(
    substring(col("value"), 1, 4).cast("int").alias("EMISSOR"),
    trim(substring(col("value"), 5, 6)).alias("ULT_PERIODO_FECH"),
    trim(substring(col("value"), 11, 8)).alias("DT_VCT"),
    substring(col("value"), 19, 4).cast("int").alias("ATRASO"),
    substring(col("value"), 23, 14).cast("bigint").alias("VALOR_1"),
    trim(substring(col("value"), 37, 1)).alias("SINAL_VALOR"),
    substring(col("value"), 38, 5).cast("int").alias("FILIAL_EMISSOR"),
    substring(col("value"), 43, 2).cast("int").alias("PRODUTO"),
    substring(col("value"), 45, 9).cast("int").alias("NUCONTA"),
    substring(col("value"), 54, 7).cast("int").alias("GRP_AFINIDADE"),
    trim(substring(col("value"), 61, 30)).alias("DESC_FILIAL"),
    trim(substring(col("value"), 91, 30)).alias("DESC_GRP_AFIN"),
    substring(col("value"), 121, 2).cast("int").alias("TIPO_CARTAO"),
    trim(substring(col("value"), 123, 30)).alias("DESC_TIPO_CARTAO"),
    substring(col("value"), 153, 2).cast("int").alias("TIPO_DOCUMENTO"),
    trim(substring(col("value"), 155, 15)).alias("NUM_DOCUMENTO"),
    trim(substring(col("value"), 170, 47)).alias("NOME_TITULAR"),
    substring(col("value"), 217, 2).cast("int").alias("CL_STATUS_1")
)

# Formatando campos e criando novos
df_basileia = df_basileia.select(
    to_date(lit(DATA_1), 'yyyyMMdd').alias("DATA_ARQ"),
    to_date(col("ULT_PERIODO_FECH"), 'yyyyMM').alias("SAFRA"),
    to_date(col("DT_VCT"), 'yyyyMMdd').alias("VENCIMENTO"),
    col("ATRASO"),
    when(col("SINAL_VALOR") == "-", col("VALOR_1") / -100.0)
        .otherwise(col("VALOR_1") / 100.0).alias("VALOR"),
    col("FILIAL_EMISSOR").alias("FILIAL"),
    col("PRODUTO"),
    col("NUCONTA"),
    lit(1).alias("X"),
    col("DESC_FILIAL").alias("GRP_AFIN"),
    col("DESC_GRP_AFIN"),
    col("TIPO_CARTAO").alias("VARIANTE"),
    col("DESC_TIPO_CARTAO").alias("DESC_VAR"),
    col("NUM_DOCUMENTO").alias("NUCPFCNPJ"),
    col("NOME_TITULAR").alias("NOME"),
    col("CL_STATUS_1")
)

# Criar tabela temporária
df_basileia.createOrReplaceTempView(f"BASILEIA_{DATA_1}")

In [ ]:
# Cell 10
spark.sql(f"""
    CREATE OR REPLACE TABLE BASILEIA_{DATA_1} AS
    SELECT
        A.*,
        B.DATA_ENQUAD,
        -- Cálculo de ATRASO_2 (ajustado com enquadramento)
        CASE 
            WHEN B.DATA_ENQUAD < DATE('2016-03-04') 
                THEN DATEDIFF(A.DATA_ARQ, DATE_SUB(B.DATA_ENQUAD, 68))
            ELSE DATEDIFF(A.DATA_ARQ, DATE_SUB(B.DATA_ENQUAD, 75))
        END AS ATRASO_2,
        
        -- DIAS_AT final
        CASE
            WHEN B.DATA_ENQUAD IS NOT NULL AND B.DATA_ENQUAD <= A.DATA_ARQ
                THEN CASE 
                    WHEN B.DATA_ENQUAD < DATE('2016-03-04') 
                        THEN DATEDIFF(A.DATA_ARQ, DATE_SUB(B.DATA_ENQUAD, 68))
                    ELSE DATEDIFF(A.DATA_ARQ, DATE_SUB(B.DATA_ENQUAD, 75))
                END
            ELSE A.ATRASO
        END AS DIAS_AT,
        
        -- DATA_INICIO_ATRASO
        CASE 
            WHEN A.ATRASO = 0 THEN A.VENCIMENTO
            ELSE DATE_SUB(A.DATA_ARQ, 
                CASE
                    WHEN B.DATA_ENQUAD IS NOT NULL AND B.DATA_ENQUAD <= A.DATA_ARQ
                        THEN CASE 
                            WHEN B.DATA_ENQUAD < DATE('2016-03-04') 
                                THEN DATEDIFF(A.DATA_ARQ, DATE_SUB(B.DATA_ENQUAD, 68))
                            ELSE DATEDIFF(A.DATA_ARQ, DATE_SUB(B.DATA_ENQUAD, 75))
                        END
                    ELSE A.ATRASO
                END
            )
        END AS DATA_INICIO_ATRASO,
        
        -- CL_STATUS
        CASE
            WHEN CASE
                    WHEN B.DATA_ENQUAD IS NOT NULL AND B.DATA_ENQUAD <= A.DATA_ARQ
                        THEN CASE 
                            WHEN B.DATA_ENQUAD < DATE('2016-03-04') 
                                THEN DATEDIFF(A.DATA_ARQ, DATE_SUB(B.DATA_ENQUAD, 68))
                            ELSE DATEDIFF(A.DATA_ARQ, DATE_SUB(B.DATA_ENQUAD, 75))
                        END
                    ELSE A.ATRASO
                END > 360 AND A.CL_STATUS_1 = 1 THEN 2
            WHEN B.DATA_ENQUAD IS NOT NULL AND B.DATA_ENQUAD <= A.DATA_ARQ 
                AND CASE
                    WHEN B.DATA_ENQUAD IS NOT NULL AND B.DATA_ENQUAD <= A.DATA_ARQ
                        THEN CASE 
                            WHEN B.DATA_ENQUAD < DATE('2016-03-04') 
                                THEN DATEDIFF(A.DATA_ARQ, DATE_SUB(B.DATA_ENQUAD, 68))
                            ELSE DATEDIFF(A.DATA_ARQ, DATE_SUB(B.DATA_ENQUAD, 75))
                        END
                    ELSE A.ATRASO
                END <= 360 THEN 1
            WHEN B.DATA_ENQUAD IS NULL OR B.DATA_ENQUAD > A.DATA_ARQ THEN 0
        END AS CL_STATUS
        
    FROM BASILEIA_{DATA_1} A
    LEFT JOIN MOVTO.BASE_HIST_ENQUAD B
        ON A.NUCONTA = B.NUCONTA
        AND A.PRODUTO = B.PRODUTO
""")

# Limpar colunas desnecessárias
spark.sql(f"""
    CREATE OR REPLACE TABLE BASILEIA_{DATA_1} AS
    SELECT
        DATA_ARQ,
        SAFRA,
        VENCIMENTO,
        DIAS_AT,
        DATA_ENQUAD,
        DATA_INICIO_ATRASO,
        FILIAL,
        NUCONTA,
        X,
        GRP_AFIN,
        DESC_GRP_AFIN,
        VARIANTE,
        DESC_VAR,
        NUCPFCNPJ,
        NOME,
        CL_STATUS,
        VALOR,
        PRODUTO
    FROM BASILEIA_{DATA_1}
""")

In [ ]:
# Cell 11
# MARCAR NO ARQUIVO DA BASILEIA SE HOUVE ENQUADRAMENTO PARA A CONTA
spark.sql(f"""
            CREATE OR REPLACE TABLE BASILEIA_{DATA_1} AS
            SELECT
                A.DATA_ARQ, 
                A.SAFRA,
                A.VENCIMENTO, 
                A.ATRASO,
                A.FILIAL, 
                A.NUCONTA,
                A.X,
                A.GRP_AFIN,
                A.DESC_GRP_AFIN,
                A.VARIANTE,
                A.DESC_VAR,
                A.NUCPFCNPJ,
                A.NOME,
                A.CL_STATUS_1,
                A.VALOR,
                A.PRODUTO,
                B.DATA_ENQUAD
            FROM BASILEIA_{DATA_1} A 
            LEFT JOIN MOVTO.BASE_HIST_ENQUAD B
            ON 
            A.NUCONTA = B.NUCONTA
            AND
            A.PRODUTO = B.PRODUTO
            GROUP BY
                1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16;
                """)

In [ ]:
# Cell 12
# ALIMENTANDO BASE DA BASILEIA COM INFORMAÇÕES DA BASE EXTRAIDA DO CYBER
spark.sql(f"""
            CREATE OR REPLACE TABLE BASILEIA_{DATA_1} AS
            SELECT
                A.DATA_ARQ, 
                A.SAFRA, 
                A.VENCIMENTO, 
                A.DIAS_AT,
                A.DATA_INICIO_ATRASO, 
                A.FILIAL,
                B.CHAVE, 
                A.NUCONTA,
                A.X,
                A.GRP_AFIN,
                A.DESC_GRP_AFIN,
                A.VARIANTE,
                A.DESC_VAR,
                A.NUCPFCNPJ,
                A.NOME,
                A.CL_STATUS,
                A.DATA_ENQUAD,	
                A.VALOR,
                A.PRODUTO,
                B.DTENVIO,
                B.DTENVIO_REAL 	AS DTENVIO_REAL_ATU,
                B.ASSESSORIA 	AS ASSESSORIA_ATU,
                B.CS,
                CASE
                    WHEN B.FAIXA_CS = 'BS' 		THEN 'BAIXO_SALDO'
                    WHEN B.FAIXA_CS = 'RUIM' 	THEN 'RUIM'
                    WHEN B.FAIXA_CS = 'REG' 	THEN 'REGULAR'
                    WHEN B.FAIXA_CS = 'BOM'		THEN 'BOM'
                END AS FAIXA_CS_ATU,
                B.STATUS_CONTA
            FROM BASILEIA_{DATA_1} A
            LEFT JOIN BASE_CYBER_{DATA_1} B
            ON 
            A.NUCONTA = B.NUCONTA
            AND
            A.PRODUTO = B.PRODUTO
            AND
            B.COD_ASSESS NOT IN (4021,4022,4023,4024,4025,4026,4027,4032)
            AND
            DATEPART(B.DTENVIO) <= A.DATA_ARQ
            AND 
            (DATEPART (B.DT_NOVA_DISPONIB) IS NULL OR DATEPART (B.DT_NOVA_DISPONIB) <= A.DATA_ARQ) /* DEIXAR APENAS CONTAS EM COBRANCA COM REGULIZACAO ANTERIOR A DATA DA BASILEIA*/
            """)

In [ ]:
# Cell 13
# CRIANDO BASE COM MAIOR DATA DE ENVIO DA CONTA E PRODUTO
spark.sql(f"""
            CREATE TABLE BASILEIA AS 
            SELECT
                NUCONTA,
                PRODUTO,
                MAX(DTENVIO) AS DTENVIO
            FROM
                BASILEIA_{DATA_1}
            GROUP BY
                NUCONTA,
                PRODUTO;
                """)

In [ ]:
# Cell 14
# LIMPANDO A BASE DEIXANDO O MAIOR ENVIO DA CONTA E PRODUTO
spark.sql(f"""
            CREATE TABLE BASILEIA_{DATA_1} AS 
            SELECT
                A.*
            FROM
                BASILEIA_{DATA_1} A 
            INNER JOIN BASILEIA B
            ON
            A.NUCONTA = B.NUCONTA
            AND
            A.PRODUTO = B.PRODUTO
            AND
            A.DTENVIO = B.DTENVIO;
            """)

In [ ]:
# Cell 15
# ATUALIZANDO INFORMAÇÕES COM BASE NA BASILEIA DO DIA ANTERIOR
spark.sql(f"""
            CREATE TABLE BASILEIA_{DATA_1} AS 
            SELECT
                A.DATA_ARQ, 
                A.SAFRA, 
                A.VENCIMENTO, 
                A.DIAS_AT,
                B.DIAS_AT AS DIAS_AT_ANT,
                A.DATA_INICIO_ATRASO, 
                A.FILIAL,
                A.CHAVE,
                A.NUCONTA,
                A.X,
                A.GRP_AFIN,
                A.DESC_GRP_AFIN,
                A.VARIANTE,
                A.DESC_VAR,
                A.NUCPFCNPJ,
                A.NOME,
                A.CL_STATUS,
                A.DATA_ENQUAD,	
                A.VALOR,
                A.PRODUTO,
                A.STATUS_CONTA,
                DATEPART(A.DTENVIO)	AS DTENVIO 		FORMAT DDMMYY10.,
                A.DTENVIO_REAL_ATU,
                B.DTENVIO_REAL 	AS DTENVIO_REAL_ANT,
                A.ASSESSORIA_ATU,	
                B.ASSESSORIA AS ASSESSORIA_ANT,
                A.CS,
                A.FAIXA_CS_ATU,
                B.FAIXA_CS AS FAIXA_CS_ANT
            FROM
                BASILEIA_{DATA_1} A 
            LEFT JOIN MOVTO.BASILEIA_{DATA_2} B
            ON
            A.NUCONTA = B.NUCONTA
            AND
            A.PRODUTO = B.PRODUTO;
            """)

In [ ]:
# Cell 16
%sql
CREATE OR REPLACE TABLE BASILEIA_{DATA_1} AS
SELECT
    * EXCEPT (ASSESSORIA, FAIXA_CS, DTENVIO_REAL),
    
    CASE 
        WHEN ASSESSORIA_ATU IN ('TOLEDO', 'PASCHOALOTTO', 'BELLINATI', 'CRC', 'PILOTO_BOM') THEN ASSESSORIA_ATU
        ELSE ASSESSORIA_ANT
    END AS ASSESSORIA,
    
    CASE 
        WHEN FAIXA_CS_ATU != '' THEN FAIXA_CS_ATU
        ELSE FAIXA_CS_ANT
    END AS FAIXA_CS,
    
    CAST(
        COALESCE(DTENVIO_REAL_ATU, DTENVIO_REAL_ANT) 
    AS DATE) AS DTENVIO_REAL

FROM 
    BASILEIA_{DATA_1};

In [ ]:
# Cell 17
spark.sql(f"""
    CREATE OR REPLACE TABLE BASILEIA_{DATA_1} AS
    SELECT
        * EXCEPT (ASSESSORIA),
        
        CASE 
            WHEN ASSESSORIA = "TOLEDO_2" THEN "TOLEDO"
            WHEN ASSESSORIA = "BELLINATI_2" THEN "BELLINATI"
            ELSE ASSESSORIA
        END AS ASSESSORIA

    FROM 
        BASILEIA_{DATA_1}
""")

In [ ]:
# Cell 18
%sql
CREATE OR REPLACE TABLE BASILEIA_{DATA_1} AS 
SELECT DISTINCT
	A.DATA_ARQ, 
	A.SAFRA, 
	A.VENCIMENTO, 
	A.DIAS_AT,
	A.DIAS_AT_ANT,
	A.DATA_INICIO_ATRASO, 
	A.FILIAL,
	A.CHAVE, 
	A.NUCONTA,
	A.X,
	A.GRP_AFIN,
	A.DESC_GRP_AFIN,
	A.VARIANTE,
	A.DESC_VAR,
	A.NUCPFCNPJ,
	A.NOME,
	A.CL_STATUS,
	A.DATA_ENQUAD,	
	A.VALOR,
	A.PRODUTO,
	A.STATUS_CONTA,
	A.DTENVIO,
	A.DTENVIO_REAL,
	A.ASSESSORIA,
	A.CS,	
	A.FAIXA_CS
FROM
	BASILEIA_{DATA_1} A;

In [ ]:
# Cell 19
%sql
CREATE OR REPLACE TABLE BASILEIA_{DATA_1} AS 
SELECT DISTINCT
	A.DATA_ARQ, 
	A.SAFRA, 
	A.VENCIMENTO, 
	A.DIAS_AT,
	A.DIAS_AT_ANT,
	CASE WHEN DIAS_AT = 0 AND DIAS_AT_ANT = 0 THEN NULL
    ELSE A.DATA_INICIO_ATRASO END AS DATA_INICIO_ATRASO, 
	A.FILIAL,
	A.CHAVE, 
	A.NUCONTA,
	A.X,
	A.GRP_AFIN,
	A.DESC_GRP_AFIN,
	A.VARIANTE,
	A.DESC_VAR,
	A.NUCPFCNPJ,
	A.NOME,
	A.CL_STATUS,
	A.DATA_ENQUAD,	
	A.VALOR,
	A.PRODUTO,
	A.STATUS_CONTA,
	CASE WHEN DIAS_AT = 0 AND DIAS_AT_ANT = 0 THEN NULL
    ELSE A.DTENVIO END AS DTENVIO,
	CASE WHEN DIAS_AT = 0 AND DIAS_AT_ANT = 0 THEN NULL
    ELSE A.DTENVIO_REAL END AS DTENVIO_REAL,
	CASE WHEN DIAS_AT = 0 AND DIAS_AT_ANT = 0 THEN NULL
    ELSE A.ASSESSORIA END AS ASSESSORIA,
	CASE WHEN DIAS_AT = 0 AND DIAS_AT_ANT = 0 THEN ''
    ELSE A.CS END AS CS,	
	CASE WHEN DIAS_AT = 0 AND DIAS_AT_ANT = 0 THEN ''
    ELSE A.FAIXA_CS END AS FAIXA_CS
FROM
	BASILEIA_{DATA_1} A;

In [ ]:
# Cell 20
%sql
CREATE TABLE MOVTO.BASILEIA_{DATA_1}
AS SELECT * FROM BASILEIA_{DATA_1};

In [ ]:
# Cell 21
%sql
DROP TABLE BASILEIA_{DATA_1};

In [ ]:
# Cell 22
spark.sql(f"""
    CREATE OR REPLACE TABLE BASE_CYBER AS 
    SELECT
        CY.SAFRA,
        CY.DT_ARQUIVO,
        CY.DTENVIO AS DTENVIO_CYBER,
        CY.DTENVIO_REAL,
        CY.DT_NOVA_DISPONIB,
        CY.DT_NOVA_DISPONIB_REAL,
        MA.DTENVIO_REAL AS DTENVIO_REAL_MA,
        CY.TPCLASSE,
        CY.CHAVE,
        CY.NUCONTA,
        CY.PRODUTO,
        BA.NUCPFCNPJ,
        CY.FAIXA_CS_ATU,
        CY.DIG_SCORE,
        CY.DIG_RAND,
        CY.PILOTO_DIGITAL,
        CY.CONTA_DIGITAL,
        CY.COD_ASSESS,
        CY.ASSESSORIA,
        MA.ASSESSORIA AS ASSESSORIA_MA,
        CASE
            WHEN MA.DIAS_AT = 0 THEN MA.VENCIMENTO
            ELSE MA.DATA_INICIO_ATRASO
        END AS DT_INICIO_ATRASO,
        CY.DIAS_AT,
        CY.ATRASO_ATU,
        CY.ATRASO_INI_MES,
        BA.DIAS_AT AS DIAS_AT_BA,
        BA.DIAS_AT_ANT AS DIAS_AT_ANT_BA,
        MA.DIAS_AT AS DIAS_AT_MA,
        MA.DIAS_AT_ANT AS DIAS_AT_ANT_MA,
        CY.SALDO_DEVEDOR_VENC,
        CY.DMCURBAL,
        CY.VL_NOVA_DIVIDA,
        MA.VALOR AS VALOR_MA,
        BA.VALOR AS VALOR_BA,
        DA.VALOR AS VALOR_BA_ANT,
        CASE
            WHEN CY.TPCLASSE = 'F' THEN CY.SALDO_DEVEDOR_VENC
            WHEN MA.VALOR <= 0 THEN CY.DMCURBAL
            ELSE MA.VALOR
        END AS VLR_COBRANCA,
        CASE
            WHEN ENQ.DATA_ENQUAD <= DATE('{DT_ARQ.strftime('%Y-%m-%d')}') THEN ENQ.DATA_ENQUAD
            ELSE NULL
        END AS DATA_ENQUAD,
        CASE
            WHEN ENQ.DATA_ENQUAD IS NULL THEN NULL
            WHEN ENQ.DATA_ENQUAD <= DATE('{DT_ARQ.strftime('%Y-%m-%d')}') THEN DA.VALOR
            ELSE NULL
        END AS VLR_ENQUADRAMENTO,
        -- PILOTO MCKINSEY INICIO JUL23
        CASE
            WHEN CY.U3SCORE4 >= 2000 AND CY.U3SCORE4 <= 2998 
                AND CY.COD_ASSESS IN (4001,4006) 
                AND CY.SAFRA IN (DATE('2023-07-01'), DATE('2023-08-01')) THEN 'S'
            WHEN CY.U3SCORE4 >= 2000 AND CY.U3SCORE4 <= 2998 
                AND CY.COD_ASSESS IN (4001) 
                AND CY.SAFRA IN (DATE('2023-09-01'), DATE('2023-10-01')) THEN 'S'
            WHEN CY.U3SCORE4 IN (2003,2030) 
                AND CY.COD_ASSESS IN (4001) 
                AND CY.SAFRA IN (DATE('2023-11-01')) THEN 'S'
            WHEN CY.U3SCORE4 IN (2003,2030) 
                AND CY.COD_ASSESS IN (4001,4006) 
                AND CY.SAFRA >= DATE('2023-12-01') THEN 'S'
            WHEN CY.U3SCORE4 IN (3003) 
                AND CY.COD_ASSESS IN (4001,4006) 
                AND CY.SAFRA >= DATE('2023-12-01') THEN 'C'
            ELSE 'N'
        END AS PILOTO_MCKINSEY
        
    FROM BASE_CYBER_{DATA_1} AS CY

    -- BASILEIA ATUAL
    LEFT JOIN MOVTO.BASILEIA_{DATA_1} AS BA
        ON CY.NUCONTA = BA.NUCONTA
        AND CY.PRODUTO = BA.PRODUTO

    -- BASILEIA DIA ANTERIOR
    LEFT JOIN MOVTO.BASILEIA_{DATA_2} AS DA
        ON CY.NUCONTA = DA.NUCONTA
        AND CY.PRODUTO = DA.PRODUTO

    -- ÚLTIMA BASILEIA DO MÊS ANTERIOR
    LEFT JOIN MOVTO.BASILEIA_{DATA_3} AS MA
        ON CY.NUCONTA = MA.NUCONTA
        AND CY.PRODUTO = MA.PRODUTO

    -- BASE DE ENQUADRAMENTO
    LEFT JOIN MOVTO.BASE_HIST_ENQUAD AS ENQ
        ON CY.NUCONTA = ENQ.NUCONTA
        AND CY.PRODUTO = ENQ.PRODUTO
""")

In [ ]:
# Cell 23
spark.sql(f"""
			CREATE TABLE BASE_CYBER_1 AS 
			SELECT distinct
				CY.SAFRA,
				CY.DT_ARQUIVO,
				CY.DTENVIO_CYBER,
				CY.DTENVIO_REAL AS DTENVIO,
				DTENVIO_REAL_MA,
				CY.DT_NOVA_DISPONIB,
				CY.DT_NOVA_DISPONIB_REAL,
				CY.TPCLASSE,
				CY.CHAVE,
				CY.NUCONTA,
				CY.PRODUTO,
				CY.NUCPFCNPJ,
				CY.FAIXA_CS_ATU,
				CY.DIG_SCORE,
				CY.DIG_RAND,
				/* MARCANDO OS CLIENTES QUE ENTRARAM NA COBRANCA DIGITAL E EXPIROU COM 30 DIAS E FOI ENVIADO PARA ASSESSORIA NORMAL*/
				CASE
					WHEN B.NUCONTA IS NULL THEN CY.PILOTO_DIGITAL
					ELSE 'S'
				END AS PILOTO_DIGITAL,
				CY.CONTA_DIGITAL,
				CY.COD_ASSESS,
				CY.ASSESSORIA,
				CY.ASSESSORIA_MA,
				CY.DT_INICIO_ATRASO,
				CY.DIAS_AT,
				CY.ATRASO_ATU,
				CY.ATRASO_INI_MES,
				CY.DIAS_AT_BA,
				CY.DIAS_AT_ANT_BA,
				CY.DIAS_AT_MA,
				CY.DIAS_AT_ANT_MA,
				CASE 
					WHEN CY.TPCLASSE = 'F'		THEN 'FLUXO'
					WHEN CY.DIAS_AT_MA <= 30	THEN 'ATE 30 DIAS DE ATRASO'
					WHEN CY.DIAS_AT_MA > 30		THEN 'ACIMA DE 30 DIAS DE ATRASO'
				END AS FX_ATRASO_2,
				CASE 
					WHEN CY.TPCLASSE = 'F'		THEN 'FLUXO'
					WHEN CY.DIAS_AT_MA <= 30	THEN 'ATE 30 DIAS DE ATRASO'
					WHEN CY.DIAS_AT_MA <= 60	THEN 'DE 31 A 60 DIAS DE ATRASO'
					WHEN CY.DIAS_AT_MA > 60		THEN 'ACIMA DE 60 DIAS DE ATRASO'
				END AS FX_ATRASO_NEW,
				CY.SALDO_DEVEDOR_VENC,
				CY.DMCURBAL,
				CY.VL_NOVA_DIVIDA,
				CY.VALOR_MA,
				CY.VALOR_BA,
				CY.VALOR_BA_ANT,
				CY.VLR_COBRANCA,
				CY.DATA_ENQUAD,
				CY.VLR_ENQUADRAMENTO,
				CASE
					WHEN CY.ATRASO_INI_MES >= 96																						THEN 'RETIRAR'  /* ATRASO ACIMA DE 75 NÃO ENTRA NA COBRANÇA */
					WHEN CY.TPCLASSE = 'E'	AND CY.DIAS_AT_MA IN(0,.)	AND CY.DIAS_AT_ANT_MA IN (0,.)									THEN 'RETIRAR'  /* ESTOQUE SEM ATRASO */
					WHEN CY.TPCLASSE = 'E'	AND CY.DIAS_AT_MA = 0 	AND CY.DIAS_AT_ANT_MA >= 0											THEN 'RETIRAR'  /* ESTOQUE PAGO NO FINAL DO MÊS */

					WHEN CY.TPCLASSE = 'F'	AND CY.DIAS_AT >= 95																		THEN 'RETIRAR'  /*ENVIO ACIMA DA REGUA DE COBRANCA */
			/*		WHEN CY.TPCLASSE = 'F'	AND CY.DIAS_AT_BA IN(0,.) 	AND CY.DIAS_AT_ANT_BA IN(0,.)									THEN 'RETIRAR'  */
					WHEN CY.TPCLASSE = 'F' AND CY.DT_ARQUIVO = CY.DTENVIO_REAL AND CY.DIAS_AT_BA IN(0,.) AND CY.DIAS_AT_ANT_BA IN (0,.) THEN 'RETIRAR'	/* ENVIO PAGO */
					WHEN CY.TPCLASSE = 'F' AND CY.DT_ARQUIVO = CY.DTENVIO_REAL AND CY.DIAS_AT_BA = 0 AND CY.DIAS_AT_ANT_BA >= 0 		THEN 'RETIRAR'	/* ENVIO PAGO */
					
					WHEN CY.DATA_ENQUAD IS NOT MISSING AND  CY.DATA_ENQUAD <= CY.DTENVIO_REAL	THEN 'RETIRAR'  /*ENQUADRAMENTO ANTES DO ENVIO / ENVIO INDEVIDO */
					
					WHEN CY.DTENVIO_REAL = CY.DT_NOVA_DISPONIB_REAL THEN 'RETIRAR'

			ELSE 'EM_COBRANCA'
				END AS TPLANCAMENTO,
				CASE
					WHEN CY.ATRASO_INI_MES >= 76																						THEN 'ATRASO_ACIMA_DE_75_DIAS'  /* ATRASO ACIMA DE 75 NÃO ENTRA NA COBRANÇA */
					WHEN CY.TPCLASSE = 'E'	AND CY.DIAS_AT_MA IN(0,.)	AND CY.DIAS_AT_ANT_MA IN (0,.)									THEN 'ESTOQUE_SEM_ATRASO'  /* ESTOQUE SEM ATRASO */
					WHEN CY.TPCLASSE = 'E'	AND CY.DIAS_AT_MA = 0 	AND CY.DIAS_AT_ANT_MA >= 0											THEN 'ESTOQUE_PAGO_NO_ULTIMO_DIA'  /* ESTOQUE PAGO NO FINAL DO MÊS */

					WHEN CY.TPCLASSE = 'F'	AND CY.DIAS_AT >= 75																		THEN 'ENVIO_ACIMA_DE_75_DIAS'  /*ENVIO ACIMA DA REGUA DE COBRANCA */
			/*		WHEN CY.TPCLASSE = 'F'	AND CY.DIAS_AT_BA IN(0,.) 	AND CY.DIAS_AT_ANT_BA IN(0,.)									THEN 'ENVIO_PAGO'  */
					WHEN CY.TPCLASSE = 'F' AND CY.DT_ARQUIVO = CY.DTENVIO_REAL AND CY.DIAS_AT_BA IN(0,.) AND CY.DIAS_AT_ANT_BA IN (0,.) THEN 'ENVIO_PAGO'	/* ENVIO PAGO */
					WHEN CY.TPCLASSE = 'F' AND CY.DT_ARQUIVO = CY.DTENVIO_REAL AND CY.DIAS_AT_BA = 0 AND CY.DIAS_AT_ANT_BA >= 0 		THEN 'ENVIO_PAGO'	/* ENVIO PAGO */
					WHEN CY.DATA_ENQUAD IS NOT MISSING AND CY.DATA_ENQUAD <= CY.DTENVIO_REAL	THEN 'ENVIO_DEPOIS_DO_ENQUADRAMENTO'  /*ENQUADRAMENTO ANTES DO ENVIO / ENVIO INDEVIDO */
					WHEN CY.DTENVIO_REAL = CY.DT_NOVA_DISPONIB_REAL THEN 'ENVIO E RETIRADA DA ASSESSORIA NO MESMO DIA '
					END AS MOTIVO_RETIRADA,
				CY.PILOTO_MCKINSEY

			FROM BASE_CYBER AS CY
			LEFT JOIN MOVTO.PERF_AMIGAVEL AS B
			ON CY.NUCONTA = B.NUCONTA
			AND CY.PRODUTO = B.PRODUTO
			AND CY.DT_INICIO_ATRASO = B.DT_INICIO_ATRASO
			AND B.PILOTO_DIGITAL = 'S'
			AND CY.COD_ASSESS NOT IN (4010)
""")

In [ ]:
# Cell 24
if not spark.catalog.tableExists(f"MOVTO.CA_CARTEIRA_{ANOMESCARGA}"):
    spark.sql(f"""
        CREATE TABLE MOVTO.CA_CARTEIRA_{ANOMESCARGA}
        AS SELECT * FROM BASE_CYBER_1
    """)

else:

    df_novos = spark.sql(f"""
        SELECT A.*
        FROM BASE_CYBER_1 A
        LEFT ANTI JOIN MOVTO.CA_CARTEIRA_{ANOMESCARGA} B
            ON A.SAFRA = B.SAFRA
            AND A.DTENVIO = B.DTENVIO
            AND A.NUCONTA = B.NUCONTA
            AND A.PRODUTO = B.PRODUTO
    """)
    

    df_novos.write.mode("append").saveAsTable(f"MOVTO.CA_CARTEIRA_{ANOMESCARGA}")


In [ ]:
# Cell 25
spark.sql(f"""
    CREATE OR REPLACE TABLE ENQUADRAMENTO AS
    SELECT
        SAFRA,
        DTENVIO,
        NUCONTA,
        PRODUTO,
        DATA_ENQUAD,
        VLR_ENQUADRAMENTO
    FROM BASE_CYBER_1
    WHERE VLR_ENQUADRAMENTO IS NOT NULL 
        AND TPLANCAMENTO != 'RETIRAR'
""")

In [ ]:
# Cell 26
# Primeiro UPDATE - VLR_ENQUADRAMENTO
spark.sql(f"""
    MERGE INTO MOVTO.CA_CARTEIRA_{ANOMESCARGA} AS A
    USING ENQUADRAMENTO AS B
    ON A.SAFRA = B.SAFRA
        AND A.DTENVIO = B.DTENVIO
        AND A.NUCONTA = B.NUCONTA
        AND A.PRODUTO = B.PRODUTO
    WHEN MATCHED AND A.DATA_ENQUAD IS NULL THEN
        UPDATE SET A.VLR_ENQUADRAMENTO = B.VLR_ENQUADRAMENTO
""")

# Segundo UPDATE - DATA_ENQUAD
spark.sql(f"""
    MERGE INTO MOVTO.CA_CARTEIRA_{ANOMESCARGA} AS A
    USING ENQUADRAMENTO AS B
    ON A.SAFRA = B.SAFRA
        AND A.DTENVIO = B.DTENVIO
        AND A.NUCONTA = B.NUCONTA
        AND A.PRODUTO = B.PRODUTO
    WHEN MATCHED AND A.DATA_ENQUAD IS NULL THEN
        UPDATE SET A.DATA_ENQUAD = B.DATA_ENQUAD
""")

In [ ]:
# Cell 27
%sql
CREAT TABLE ENQUADRAMENTO AS
SELECT
	SAFRA,
	DTENVIO,
	NUCONTA,
	PRODUTO,
	DATA_ENQUAD,
	VLR_ENQUADRAMENTO;
FROM BASE_CYBER_1;
WHERE VLR_ENQUADRAMENTO IS NOT NULL AND TPLANCAMENTO != 'RETIRAR';
KEEP 
RUN;

In [ ]:
# Cell 28
spark.sql(f"""
			UPDATE MOVTO.CA_CARTEIRA_{ANOMESCARGA} A
					SET VLR_ENQUADRAMENTO = 
				(SELECT
					VLR_ENQUADRAMENTO
						FROM ENQUADRAMENTO B
							WHERE a.SAFRA = b.SAFRA
								AND A.DTENVIO = B.DTENVIO
								AND A.NUCONTA = B.NUCONTA
								AND A.PRODUTO = B.PRODUTO)
					WHERE A.DATA_ENQUAD = IS NULL;


				UPDATE MOVTO.CA_CARTEIRA_{ANOMESCARGA} A
					SET DATA_ENQUAD = 
				(SELECT
					DATA_ENQUAD
						FROM ENQUADRAMENTO b
							WHERE a.SAFRA = b.SAFRA
								AND A.DTENVIO = B.DTENVIO
								AND A.NUCONTA = B.NUCONTA
								AND A.PRODUTO = B.PRODUTO)
					WHERE A.DATA_ENQUAD IS NULL;
		""")

In [ ]:
# Cell 29
%sql
CREATE TABLE WORK.FILTER_FOR_BASE AS 
   SELECT t1.NUCONTA, 
          t1.PRODUTO, 
          t1.DATA_ENQUAD
      FROM WORK.BASE t1
      ORDER BY t1.DATA_ENQUAD;

In [ ]:
# Cell 30
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW BA_REGULARIZADO AS
    SELECT * FROM BASE_CYBER_1
    WHERE TPLANCAMENTO != 'RETIRAR' 
        AND DATA_ENQUAD IS NULL
""")

In [ ]:
# Cell 31
%sql
CREATE TABLE BA_REGULARIZADO_1 AS
SELECT
	REG.SAFRA,
	REG.CHAVE,
	REG.NUCONTA,
	REG.PRODUTO,
	REG.NUCPFCNPJ,
	REG.DTENVIO,
	REG.DT_NOVA_DISPONIB,
	REG.DT_NOVA_DISPONIB_REAL,
	REG.COD_ASSESS,
	REG.DT_ARQUIVO AS DT_REGULARIZADO,
	C.DU AS DU_REGULARIZADO,
	CASE
		WHEN REG.VLR_COBRANCA < REG.VALOR_BA_ANT THEN REG.VALOR_BA_ANT
		ELSE REG.VLR_COBRANCA
	END AS VLR_REGULARIZADO,
	REG.VLR_COBRANCA,
	REG.VALOR_BA,
	REG.VALOR_BA_ANT

FROM BA_REGULARIZADO AS REG 

LEFT JOIN b_geral.DIAS_UTEIS C
	ON REG.DT_ARQUIVO = C.DATA

WHERE 
	(REG.DIAS_AT_BA IN(0,.)  OR DIAS_AT_BA < REG.DIAS_AT_ANT_BA)
	AND (REG.DIAS_AT_ANT_BA > 0 OR REG.DIAS_AT_ANT_BA = .)
/*	AND (REG.DT_ARQUIVO >= REG.DTENVIO AND REG.DT_ARQUIVO <= REG.DT_NOVA_DISPONIB_REAL )  */

In [ ]:
# Cell 32
%sql
CREATE OR REPLACE TABLE BA_REGULARIZADO_2 AS
SELECT 
    A.*
FROM 
    BA_REGULARIZADO_1 AS A
LEFT JOIN 
    MOVTO.BASE_REGULARIZADO AS B
ON 
    A.SAFRA = B.SAFRA
    AND A.DTENVIO = B.DTENVIO
    AND A.NUCONTA = B.NUCONTA
    AND A.PRODUTO = B.PRODUTO
WHERE
    B.SAFRA IS NULL;

In [ ]:
# Cell 33
%sql
INSERT INTO MOVTO.BASE_REGULARIZADO
SELECT * FROM BA_REGULARIZADO_2;

In [ ]:
# Cell 34
%sql
CREATE OR REPLACE TABLE ANALISE_REGULARIZADO AS
SELECT * FROM BASE_CYBER_1
WHERE
    TPLANCAMENTO != 'RETIRAR'
    AND DATA_ENQUAD IS NULL
    AND DIAS_AT_ANT_BA > DIAS_AT_BA;

In [ ]:
# Cell 35
spark.sql(f"""
			CREATE TABLE ANALISE_REGULARIZADO_1 AS
			SELECT
				A.*
			FROM ANALISE_REGULARIZADO AS A

			LEFT JOIN MOVTO.CA_CARTEIRA_{ANOMESCARGA} AS B
			ON A.NUCONTA = B.NUCONTA
			AND A.PRODUTO = B.PRODUTO

			LEFT JOIN MOVTO.BASE_REGULARIZADO AS C
			ON A.NUCONTA = C.NUCONTA
			AND A.PRODUTO = C.PRODUTO
			AND C.SAFRA = {SAFRA}

			WHERE B.TPLANCAMENTO = 'EM_COBRANCA'
			AND C.NUCONTA IS NULL
			AND (A.DIAS_AT_BA IS NULL OR A.DIAS_AT_BA = 0)
""")

In [ ]:
# Cell 36
%sql
CREATE TABLE ANALISE_REGULARIZADO_2 AS 
SELECT
	REG.SAFRA,
	REG.CHAVE,
	REG.NUCONTA,
	REG.PRODUTO,
	REG.NUCPFCNPJ,
	REG.DTENVIO,
	REG.DT_NOVA_DISPONIB,
	REG.DT_NOVA_DISPONIB_REAL,
	REG.COD_ASSESS,
	CAST(REG.DT_ARQUIVO AS DATE) AS DT_REGULARIZADO,
	C.DU AS DU_REGULARIZADO,
	CASE
		WHEN REG.VLR_COBRANCA < REG.VALOR_BA_ANT THEN REG.VALOR_BA_ANT
		ELSE REG.VLR_COBRANCA
	END AS VLR_REGULARIZADO,
	REG.VLR_COBRANCA,
	REG.VALOR_BA,
	REG.VALOR_BA_ANT

FROM ANALISE_REGULARIZADO_1 AS REG 

LEFT JOIN b_geral.DIAS_UTEIS C
	ON REG.DT_ARQUIVO = C.DATA

In [ ]:
# Cell 37
%sql
INSERT INTO MOVTO.BASE_REGULARIZADO
SELECT * FROM ANALISE_REGULARIZADO_2;

In [ ]:
# Cell 38
%sql
CREATE OR REPLACE TABLE ANALISE_REGULARIZADO_3 AS
SELECT * FROM BASE_CYBER_1
WHERE
    TPLANCAMENTO != 'RETIRAR'
    AND DT_ARQUIVO = DT_NOVA_DISPONIB;

In [ ]:
# Cell 39
spark.sql(f"""
			CREATE TABLE ANALISE_REGULARIZADO_4 AS
			SELECT
				A.*
			FROM ANALISE_REGULARIZADO_3 AS A
			LEFT JOIN MOVTO.CA_CARTEIRA_{ANOMESCARGA} AS B
			ON A.NUCONTA = B.NUCONTA
			AND A.PRODUTO = B.PRODUTO
			LEFT JOIN MOVTO.BASE_REGULARIZADO AS C
			ON A.NUCONTA = C.NUCONTA
			AND A.PRODUTO = C.PRODUTO
			AND C.SAFRA = {SAFRA}
			WHERE B.TPLANCAMENTO = 'EM_COBRANCA'
			AND C.NUCONTA IS NULL
			AND (A.DIAS_AT_BA IS NULL OR A.DIAS_AT_BA = 0))
""")

In [ ]:
# Cell 40
%sql
CREATE TABLE ANALISE_REGULARIZADO_5 AS 
SELECT
	REG.SAFRA,
	REG.CHAVE,
	REG.NUCONTA,
	REG.PRODUTO,
	REG.NUCPFCNPJ,
	REG.DTENVIO,
	REG.DT_NOVA_DISPONIB,
	REG.DT_NOVA_DISPONIB_REAL,
	REG.COD_ASSESS,
    CAST(REG.DT_ARQUIVO AS DATE) AS DT_REGULARIZADO
	C.DU AS DU_REGULARIZADO,
	CASE
		WHEN REG.VLR_COBRANCA < REG.VALOR_BA_ANT THEN REG.VALOR_BA_ANT
		ELSE REG.VLR_COBRANCA
	END AS VLR_REGULARIZADO,
	REG.VLR_COBRANCA,
	REG.VALOR_BA,
	REG.VALOR_BA_ANT
FROM ANALISE_REGULARIZADO_4 AS REG 
LEFT JOIN b_geral.DIAS_UTEIS C
	ON REG.DT_ARQUIVO = C.DATA

In [ ]:
# Cell 41
spark.sql(f"""
    DELETE FROM MOVTO.BASE_REGULARIZADO
    WHERE DT_REGULARIZADO > DATE('{DATA_REG}')
""")

In [ ]:
# Cell 42
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW BA_REGULARIZADO AS
    SELECT * FROM BASE_CYBER_1
    WHERE TPLANCAMENTO != 'RETIRAR' 
        AND DATA_ENQUAD IS NULL
""")

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW BA_REGULARIZADO_1 AS
    SELECT
        REG.SAFRA,
        REG.CHAVE,
        REG.NUCONTA,
        REG.PRODUTO,
        REG.NUCPFCNPJ,
        REG.DTENVIO,
        REG.DT_NOVA_DISPONIB,
        REG.DT_NOVA_DISPONIB_REAL,
        REG.COD_ASSESS,
        REG.DT_ARQUIVO AS DT_REGULARIZADO,
        C.DU AS DU_REGULARIZADO,
        CASE
            WHEN REG.VLR_COBRANCA < REG.VALOR_BA_ANT THEN REG.VALOR_BA_ANT
            ELSE REG.VLR_COBRANCA
        END AS VLR_REGULARIZADO,
        REG.VLR_COBRANCA,
        REG.VALOR_BA,
        REG.VALOR_BA_ANT
    FROM BA_REGULARIZADO AS REG 
    LEFT JOIN b_geral.DIAS_UTEIS C
        ON REG.DT_ARQUIVO = C.DATA
    WHERE 
        ((REG.DIAS_AT_BA IS NULL OR REG.DIAS_AT_BA = 0) 
            OR REG.DIAS_AT_BA < REG.DIAS_AT_ANT_BA)
        AND (REG.DIAS_AT_ANT_BA > 0 OR REG.DIAS_AT_ANT_BA IS NULL)
""")


In [ ]:
# Cell 43
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW BA_REGULARIZADO_2 AS
    SELECT A.*
    FROM BA_REGULARIZADO_1 AS A
    LEFT ANTI JOIN MOVTO.BASE_REGULARIZADO AS B
        ON A.SAFRA = B.SAFRA
        AND A.DTENVIO = B.DTENVIO
        AND A.NUCONTA = B.NUCONTA
        AND A.PRODUTO = B.PRODUTO
""")

spark.sql(f"""
    INSERT INTO MOVTO.BASE_REGULARIZADO
    SELECT * FROM BA_REGULARIZADO_2
""")

In [ ]:
# Cell 44
%sql
// VERIFICAR
DELETE FROM MOVTO.BASE_REGULARIZADO
WHERE DT_REGULARIZADO > {DATA_REG};

CREAT OR REPLACE TABLE MOVTO.BASE_REGULARIZADO AS
SELECT * 
FROM MOVTO.BASE_REGULARIZADO
WHERE DT_REGULARIZADO <= {DATA_REG};

In [ ]:
# Cell 45
spark.sql(f"""
    DELETE FROM MOVTO.PERF_AMIGAVEL
    WHERE SAFRA = DATE('{DTINICIOMES.strftime('%Y-%m-%d')}')
""")

In [ ]:
# Cell 46
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW ANALISE_REGULARIZADO AS
    SELECT * FROM BASE_CYBER_1
    WHERE TPLANCAMENTO != 'RETIRAR' 
        AND DATA_ENQUAD IS NULL
        AND DIAS_AT_ANT_BA > DIAS_AT_BA
""")


spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW ANALISE_REGULARIZADO_1 AS
    SELECT A.*
    FROM ANALISE_REGULARIZADO AS A
    INNER JOIN MOVTO.CA_CARTEIRA_{ANOMESCARGA} AS B
        ON A.NUCONTA = B.NUCONTA
        AND A.PRODUTO = B.PRODUTO
    LEFT ANTI JOIN (
        SELECT NUCONTA, PRODUTO, SAFRA
        FROM MOVTO.BASE_REGULARIZADO
        WHERE SAFRA = DATE('{SAFRA.strftime('%Y-%m-%d')}')
    ) AS C
        ON A.NUCONTA = C.NUCONTA
        AND A.PRODUTO = C.PRODUTO
    WHERE B.TPLANCAMENTO = 'EM_COBRANCA'
        AND (A.DIAS_AT_BA IS NULL OR A.DIAS_AT_BA = 0)
""")

In [ ]:
# Cell 47
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW ANALISE_REGULARIZADO_2 AS 
    SELECT
        REG.SAFRA, REG.CHAVE, REG.NUCONTA, REG.PRODUTO,
        REG.NUCPFCNPJ, REG.DTENVIO, REG.DT_NOVA_DISPONIB,
        REG.DT_NOVA_DISPONIB_REAL, REG.COD_ASSESS,
        REG.DT_ARQUIVO AS DT_REGULARIZADO,
        C.DU AS DU_REGULARIZADO,
        CASE
            WHEN REG.VLR_COBRANCA < REG.VALOR_BA_ANT THEN REG.VALOR_BA_ANT
            ELSE REG.VLR_COBRANCA
        END AS VLR_REGULARIZADO,
        REG.VLR_COBRANCA, REG.VALOR_BA, REG.VALOR_BA_ANT
    FROM ANALISE_REGULARIZADO_1 AS REG 
    LEFT JOIN b_geral.DIAS_UTEIS C
        ON REG.DT_ARQUIVO = C.DATA
""")
    
spark.sql(f"""
    INSERT INTO MOVTO.BASE_REGULARIZADO
    SELECT * FROM ANALISE_REGULARIZADO_2
""")

In [ ]:
# Cell 48
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW ANALISE_REGULARIZADO_3 AS
    SELECT * FROM BASE_CYBER_1
    WHERE TPLANCAMENTO != 'RETIRAR' 
        AND DT_ARQUIVO = DT_NOVA_DISPONIB
""")

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW ANALISE_REGULARIZADO_4 AS
    SELECT A.*
    FROM ANALISE_REGULARIZADO_3 AS A
    INNER JOIN MOVTO.CA_CARTEIRA_{ANOMESCARGA} AS B
        ON A.NUCONTA = B.NUCONTA AND A.PRODUTO = B.PRODUTO
    LEFT ANTI JOIN (
        SELECT NUCONTA, PRODUTO FROM MOVTO.BASE_REGULARIZADO
        WHERE SAFRA = DATE('{SAFRA.strftime('%Y-%m-%d')}')
    ) AS C
        ON A.NUCONTA = C.NUCONTA AND A.PRODUTO = C.PRODUTO
    WHERE B.TPLANCAMENTO = 'EM_COBRANCA'
        AND (A.DIAS_AT_BA IS NULL OR A.DIAS_AT_BA = 0)
""")

spark.sql(f"""
    INSERT INTO MOVTO.BASE_REGULARIZADO
    SELECT
        SAFRA, CHAVE, NUCONTA, PRODUTO, NUCPFCNPJ,
        DTENVIO, DT_NOVA_DISPONIB, DT_NOVA_DISPONIB_REAL,
        COD_ASSESS, DT_ARQUIVO AS DT_REGULARIZADO,
        NULL AS DU_REGULARIZADO,
        CASE WHEN VLR_COBRANCA < VALOR_BA_ANT THEN VALOR_BA_ANT
                ELSE VLR_COBRANCA END AS VLR_REGULARIZADO,
        VLR_COBRANCA, VALOR_BA, VALOR_BA_ANT
    FROM ANALISE_REGULARIZADO_4
""")


In [ ]:
# Cell 49
spark.sql(f"""
    DELETE FROM MOVTO.BASE_REGULARIZADO
    WHERE DT_REGULARIZADO > DATE('{DATA_REG}')
""")


spark.sql("DROP VIEW IF EXISTS BA_REGULARIZADO;")
spark.sql("DROP VIEW IF EXISTS BA_REGULARIZADO_1;")
spark.sql("DROP VIEW IF EXISTS BA_REGULARIZADO_2;")
spark.sql("DROP VIEW IF EXISTS ANALISE_REGULARIZADO;")
spark.sql("DROP VIEW IF EXISTS ANALISE_REGULARIZADO_1;")
spark.sql("DROP VIEW IF EXISTS ANALISE_REGULARIZADO_2;")
spark.sql("DROP VIEW IF EXISTS ANALISE_REGULARIZADO_3;")
spark.sql("DROP VIEW IF EXISTS ANALISE_REGULARIZADO_4;")

In [ ]:
# Cell 50
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW PERFORMANCE AS 
    SELECT DISTINCT
        BA.SAFRA, BA.DT_ARQUIVO, BA.DTENVIO_CYBER, BA.DTENVIO,
        BA.DT_NOVA_DISPONIB,
        CASE WHEN BA.TPCLASSE = 'F' THEN DU.DU END AS DU_ENVIO,
        BA.TPCLASSE, BA.CHAVE, BA.NUCONTA, BA.PRODUTO, BA.NUCPFCNPJ,
        BA.FAIXA_CS_ATU, BA.DIG_SCORE, BA.DIG_RAND,
        BA.PILOTO_DIGITAL, BA.CONTA_DIGITAL, BA.COD_ASSESS, BA.ASSESSORIA,
        BA.DT_INICIO_ATRASO,
        CASE WHEN BA.TPCLASSE = 'F' THEN BA.DIAS_AT 
             ELSE BA.DIAS_AT_MA END AS DIAS_ATRASO,
        BA.FX_ATRASO_2, BA.FX_ATRASO_NEW, BA.VLR_COBRANCA,
        REG.DT_REGULARIZADO, REG.DU_REGULARIZADO, REG.VLR_REGULARIZADO,
        BA.DATA_ENQUAD, BA.VLR_ENQUADRAMENTO,
        CASE WHEN BA.DTENVIO = REG.DT_REGULARIZADO THEN 'S' 
             ELSE 'N' END AS ENVIO_PAGO,
        BA.PILOTO_MCKINSEY
    FROM MOVTO.CA_CARTEIRA_{ANOMESCARGA} AS BA
    LEFT JOIN b_geral.DIAS_UTEIS AS DU ON BA.DTENVIO = DU.DATA
    LEFT JOIN MOVTO.BASE_REGULARIZADO AS REG
        ON BA.SAFRA = REG.SAFRA
        AND BA.DTENVIO = REG.DTENVIO
        AND BA.NUCONTA = REG.NUCONTA
        AND BA.PRODUTO = REG.PRODUTO
    WHERE BA.SAFRA = DATE('{DTINICIOMES.strftime('%Y-%m-%d')}')
        AND BA.TPLANCAMENTO != 'RETIRAR'
""")

In [ ]:
# Cell 51
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW PERFORMANCE_2 AS 
    SELECT * FROM PERFORMANCE
    WHERE ENVIO_PAGO = 'N'
""")

In [ ]:
# Cell 52
spark.sql(f"""
    DELETE FROM MOVTO.PERF_AMIGAVEL
    WHERE SAFRA = DATE('{DTINICIOMES.strftime('%Y-%m-%d')}')
""")

In [ ]:
# Cell 53
spark.sql(f"""
    INSERT INTO MOVTO.PERF_AMIGAVEL
    SELECT * FROM PERFORMANCE_2
""")

In [ ]:
# Cell 54
spark.sql(f"""
    CREATE OR REPLACE TABLE MOVTO.PERF_AMIGAVEL AS
    SELECT
        * EXCEPT (DT_INICIO_ATRASO, VLR_REG_PA_FIXA),
        COALESCE(DT_INICIO_ATRASO, DATE_SUB(DTENVIO, DIAS_ATRASO)) AS DT_INICIO_ATRASO,
        CASE 
            WHEN SAFRA >= DATE('2024-03-01') 
                AND DT_REGULARIZADO IS NOT NULL
                AND DATEDIFF(DT_REGULARIZADO, 
                     COALESCE(DT_INICIO_ATRASO, DATE_SUB(DTENVIO, DIAS_ATRASO))) > 30
                THEN NULL
            ELSE VLR_REGULARIZADO
        END AS VLR_REG_PA_FIXA
    FROM MOVTO.PERF_AMIGAVEL
""")

In [ ]:
# Cell 55


In [ ]:
# Cell 56
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW BASE_FINAL_1 AS

    SELECT
        A.SAFRA, DATE('{DT_ARQ.strftime('%Y-%m-%d')}') AS DT_REF,
        DU.DU AS DU_DT_REF, CAST(NULL AS INT) AS DU_ENVIO,
        A.TPCLASSE, CAST(NULL AS STRING) AS FAIXA_CS_ATU,
        A.ASSESSORIA, A.FX_ATRASO_2, A.FX_ATRASO_NEW, A.PILOTO_MCKINSEY,
        CAST(NULL AS BIGINT) AS QTD_COBRANCA,
        CAST(NULL AS DOUBLE) AS VLR_COBRANCA,
        CAST(NULL AS BIGINT) AS QTD_REGULARIZADO,
        CAST(NULL AS DOUBLE) AS VLR_REGULARIZADO,
        CAST(NULL AS BIGINT) AS QTD_ENQUADRAMENTO,
        CAST(NULL AS DOUBLE) AS VLR_ENQUADRAMENTO,
        AVG(A.VLR_COBRANCA) AS MED_VLR_COBRANCA,
        AVG(A.DIAS_ATRASO) AS MED_DIAS_AT
    FROM MOVTO.PERF_AMIGAVEL AS A 
    LEFT JOIN b_geral.DIAS_UTEIS AS DU 
        ON DATE('{DT_ARQ.strftime('%Y-%m-%d')}') = DU.DATA
    WHERE A.SAFRA = DATE('{DTINICIOMES.strftime('%Y-%m-%d')}')
        AND A.COD_ASSESS NOT IN (4010)
    GROUP BY A.SAFRA, DT_REF, DU_DT_REF, A.TPCLASSE, 
             A.ASSESSORIA, A.FX_ATRASO_2, A.FX_ATRASO_NEW, A.PILOTO_MCKINSEY
    
    UNION ALL
    
    SELECT
        A.SAFRA, DATE('{DT_ARQ.strftime('%Y-%m-%d')}') AS DT_REF,
        DU.DU AS DU_DT_REF, CAST(NULL AS INT) AS DU_ENVIO,
        A.TPCLASSE, CAST(NULL AS STRING) AS FAIXA_CS_ATU,
        'BV' AS ASSESSORIA, A.FX_ATRASO_2, A.FX_ATRASO_NEW, A.PILOTO_MCKINSEY,
        CAST(NULL AS BIGINT) AS QTD_COBRANCA,
        CAST(NULL AS DOUBLE) AS VLR_COBRANCA,
        CAST(NULL AS BIGINT) AS QTD_REGULARIZADO,
        CAST(NULL AS DOUBLE) AS VLR_REGULARIZADO,
        CAST(NULL AS BIGINT) AS QTD_ENQUADRAMENTO,
        CAST(NULL AS DOUBLE) AS VLR_ENQUADRAMENTO,
        AVG(A.VLR_COBRANCA) AS MED_VLR_COBRANCA,
        AVG(A.DIAS_ATRASO) AS MED_DIAS_AT
    FROM MOVTO.PERF_AMIGAVEL AS A 
    LEFT JOIN b_geral.DIAS_UTEIS AS DU 
        ON DATE('{DT_ARQ.strftime('%Y-%m-%d')}') = DU.DATA
    WHERE A.SAFRA = DATE('{DTINICIOMES.strftime('%Y-%m-%d')}')
        AND A.COD_ASSESS NOT IN (4010)
    GROUP BY A.SAFRA, DT_REF, DU_DT_REF, A.TPCLASSE, 
             A.FX_ATRASO_2, A.FX_ATRASO_NEW, A.PILOTO_MCKINSEY
""")

In [ ]:
# Cell 57
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW BASE_FINAL_2 AS 
    SELECT * FROM BASE_FINAL 
    UNION ALL
    SELECT * FROM BASE_FINAL_1
""")

In [ ]:
# Cell 58
spark.sql(f"""
    DELETE FROM MOVTO.MIS_AMIGAVEL
    WHERE DT_REF = DATE('{DT_ARQ.strftime('%Y-%m-%d')}')
""")

In [ ]:
# Cell 59
spark.sql(f"""
    INSERT INTO MOVTO.MIS_AMIGAVEL
    SELECT * FROM BASE_FINAL_2
""")

In [ ]:
# Cell 60
spark.sql(f"""
    CREATE OR REPLACE TABLE CARTAO.TBAMIGAVEL_{ANOMESCARGA} AS 
    SELECT DISTINCT
        A.NUCONTA, A.PRODUTO, 'REGULARIZACAO' AS NMORIGEM,
        A.DT_REGULARIZADO AS DATA_ARQ, A.DU_REGULARIZADO AS DU,
        A.DTENVIO, A.VALOR_BA, A.VALOR_BA_ANT,
        B.ASSESSORIA, B.FX_ATRASO_NEW AS PERFIL,
        B.DT_INICIO_ATRASO AS VENCIMENTO,
        B.DIAS_AT_BA AS DIAS_AT, B.DIAS_AT_ANT_BA AS DIAS_AN,
        A.VALOR_BA AS VALOR_ATUAL, A.VALOR_BA_ANT AS VALOR_ANTERIOR
    FROM MOVTO.BASE_REGULARIZADO AS A
    LEFT JOIN MOVTO.CA_CARTEIRA_{ANOMESCARGA} AS B
        ON A.NUCONTA = B.NUCONTA
        AND A.PRODUTO = B.PRODUTO
        AND A.DTENVIO = B.DTENVIO
    WHERE A.SAFRA = DATE('{SAFRA.strftime('%Y-%m-%d')}')
        AND A.DT_REGULARIZADO IS NOT NULL
""")

In [ ]:
# Cell 61
try:
    caminho_tableau = "/mnt/tableau/"
    caminho = f"{caminho_tableau}/MIS_AMIGAVEL.csv"
    
    df = spark.table("MOVTO.MIS_AMIGAVEL")
    df.coalesce(1).write \
        .format("csv") \
        .mode("overwrite") \
        .option("header", "true") \
        .save(caminho + "_temp")
    
    # Renomear arquivo part
    part_file = [f.path for f in dbutils.fs.ls(caminho + "_temp") 
                 if f.name.startswith("part-")][0]
    dbutils.fs.mv(part_file, caminho)
    dbutils.fs.rm(caminho + "_temp", recurse=True)
    
except Exception as e:
    print(f"Error on export")

In [ ]:
# Cell 62
spark.sql(f"DROP TABLE IF EXISTS CARTAO.TBAMIGAVEL_{ANOMESCARGA}")

In [ ]:
# Cell 63
tabela = "MOVTO.MIS_AMIGAVEL"
caminho_destino = "/dbfs/Bvnet/global/DTEC/Desenvolvimento/Tableau/CobrancaVarejo/Acordo e Cartão Amigável/"
nome_arquivo = "MIS_AMIGAVEL.csv"

df = spark.table(tabela)

df.coalesce(1) \
  .write \
  .format("csv") \
  .mode("overwrite") \
  .option("header", "true") \
  .option("delimiter", ",") \
  .save(caminho_destino + "temp_export")

part_file = [f.path for f in dbutils.fs.ls(caminho_destino + "temp_export") if f.name.startswith("part-")][0]
dbutils.fs.mv(part_file, caminho_destino + nome_arquivo, recurse=True)
dbutils.fs.rm(caminho_destino + "temp_export", recurse=True)

In [ ]:
# Cell 64
%sql
CREATE OR REPLACE VIEW TBAMIGAVEL_2025101638 AS
SELECT * FROM CARTAO.TBAMIGAVEL_202510;

In [ ]:
# Cell 65
%sql
CREATE OR REPLACE VIEW WORK.BASE_HIST_ENQUAD1636 AS
SELECT * FROM MOVTO.BASE_HIST_ENQUAD;

In [ ]:
# Cell 66
spark.sql(f"""
			CREATE TABLE VALIDA AS
			SELECT
				A.*,
				B.DIAS_AT AS DIAS_BASILEIA,
				B.DIAS_AT_ANT AS DIAS_BASILEIA_ANT

			FROM MOVTO.PERF_AMIGAVEL AS A
			LEFT JOIN MOVTO.BASILEIA_{DATA_1} AS B
			ON A.NUCONTA = B.NUCONTA
			AND A.PRODUTO = B.PRODUTO
			WHERE A.SAFRA = {SAFRA}
			AND A.DT_REGULARIZADO IS NULL
			AND A.DATA_ENQUAD IS NULL

			AND (B.DIAS_AT <= 0 OR B.DIAS_AT = .)
			AND A.DT_NOVA_DISPONIB >= {DT_ARQ}

			ORDER BY
			B.DIAS_AT;
""")

In [ ]:
# Cell 67
%sql
CREATE OR REPLACE TABLE VALIDA_1 AS
SELECT * FROM BASE_CYBER_1
WHERE
    DT_ARQUIVO = DTENVIO
    AND TPLANCAMENTO = 'EM_COBRANCA'
    AND DIAS_AT_BA IS NULL;

In [ ]:
# Cell 68
%sql
CREATE TABLE WORK.FILTER_FOR_BASE_HIST_ENQUAD_0000 AS 
   SELECT t1.NUCONTA, 
          t1.PRODUTO, 
          t1.DATA_ENQUAD
      FROM MOVTO.BASE_HIST_ENQUAD t1
      ORDER BY t1.DATA_ENQUAD;

In [ ]:
# Cell 69
# Export para Power BI - usar DBFS
tabela = "MOVTO.MIS_AMIGAVEL"
caminho_destino = "/dbfs/mnt/powerbi/CobrancaVarejo/Acordo_Cartao_Amigavel/"
nome_arquivo = "MIS_AMIGAVEL.csv"

df = spark.table(tabela)

df.coalesce(1) \
  .write \
  .format("csv") \
  .mode("overwrite") \
  .option("header", "true") \
  .option("delimiter", ",") \
  .save(caminho_destino + "temp_export")

# Renomear arquivo part para nome final
part_file = [f.path for f in dbutils.fs.ls(caminho_destino + "temp_export") if f.name.startswith("part-")][0]
dbutils.fs.mv(part_file, caminho_destino + nome_arquivo)
dbutils.fs.rm(caminho_destino + "temp_export", recurse=True)

print(f"Arquivo exportado para: {caminho_destino + nome_arquivo}")

In [ ]:
# Cell 70
%sql
CREATE OR REPLACE VIEW WORK.FILTER_FOR_BASE_HIST_ENQUAD_1638 AS 
SELECT * FROM WORK.FILTER_FOR_BASE_HIST_ENQUAD_0000;

In [ ]:
# Cell 71
tabela = "MOVTO.MIS_AMIGAVEL"
caminho_destino = "N:\DVMD\Cobrança\MIS\09_ Power BI\Acordo e Cartão Amigável\"
nome_arquivo = "MIS_AMIGAVEL.csv"

df = spark.table(tabela)

df.coalesce(1) \
  .write \
  .format("csv") \
  .mode("overwrite") \
  .option("header", "true") \
  .option("delimiter", ",") \
  .save(caminho_destino + "temp_export")

part_file = [f.path for f in dbutils.fs.ls(caminho_destino + "temp_export") if f.name.startswith("part-")][0]
dbutils.fs.mv(part_file, caminho_destino + nome_arquivo, recurse=True)
dbutils.fs.rm(caminho_destino + "temp_export", recurse=True)

In [ ]:
# Cell 72
%sql
CREATE OR REPLACE VIEW WORK.BASE_FINAL_11637 AS 
SELECT * FROM WORK.BASE_FINAL_1;

In [ ]:
# Cell 73
%sql
CREATE OR REPLACE VIEW WORK.BASE_FINAL_21637 AS 
SELECT * FROM WORK.BASE_FINAL_2;

In [ ]:
# Cell 74
%sql
CREATE OR REPLACE VIEW WORK.PERFORMANCE1637 AS 
SELECT * FROM WORK.PERFORMANCE;

In [ ]:
# Cell 75
%sql
CREATE OR REPLACE VIEW WORK.BASE_REGULARIZADO1635 AS 
SELECT * FROM MOVTO.BASE_REGULARIZADO;

In [ ]:
# Cell 76


In [ ]:
# Cell 77
%sql
CREATE OR REPLACE VIEW WORK.VALIDA1638 AS
SELECT * FROM WORK.VALIDA;

In [ ]:
# Cell 78
%sql
CREATE TABLE WORK.FILTER_FOR_BASE_REGULARIZADO AS 
   SELECT t1.SAFRA, 
          t1.CHAVE, 
          t1.NUCONTA, 
          t1.PRODUTO, 
          t1.NUCPFCNPJ, 
          t1.DTENVIO, 
          t1.DT_NOVA_DISPONIB, 
          t1.COD_ASSESS, 
          t1.DT_REGULARIZADO, 
          t1.DU_REGULARIZADO, 
          t1.VLR_REGULARIZADO, 
          t1.VLR_COBRANCA, 
          t1.VALOR_BA, 
          t1.VALOR_BA_ANT, 
          t1.DT_NOVA_DISPONIB_REAL
      FROM MOVTO.BASE_REGULARIZADO t1
      ORDER BY t1.DT_REGULARIZADO;

In [ ]:
# Cell 79
%sql
CREATE OR REPLACE VIEW WORK.FILTER_FOR_BASE_REGULARIZADO1638 AS 
SELECT * FROM WORK.FILTER_FOR_BASE_REGULARIZADO;